<a href="https://colab.research.google.com/github/PRITHEW/DeepLearning_Class/blob/main/Rnn_dl_prithew.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Step 1: Install and Load Dataset

In [14]:
!pip install datasets

from datasets import load_dataset
import pandas as pd

#load the ag_news dataset from hugging face
raw_dataset = load_dataset("wangrongsheng/ag_news")

#convert the trainning split to a pandas dataframe for easy preprocessing
df=pd.DataFrame(raw_dataset['train'])
print(df.head)

<bound method NDFrame.head of                                                      text  label
0       Wall St. Bears Claw Back Into the Black (Reute...      2
1       Carlyle Looks Toward Commercial Aerospace (Reu...      2
2       Oil and Economy Cloud Stocks' Outlook (Reuters...      2
3       Iraq Halts Oil Exports from Main Southern Pipe...      2
4       Oil prices soar to all-time record, posing new...      2
...                                                   ...    ...
119995  Pakistan's Musharraf Says Won't Quit as Army C...      0
119996  Renteria signing a top-shelf deal Red Sox gene...      1
119997  Saban not going to Dolphins yet The Miami Dolp...      1
119998  Today's NFL games PITTSBURGH at NY GIANTS Time...      1
119999  Nets get Carter from Raptors INDIANAPOLIS -- A...      1

[120000 rows x 2 columns]>


Step 2: Clean and Normalize Text

In [15]:
import re  #Imports Python's Regular Expression module(used for text cleaning)

def clean_text(text):
    text = text.lower()                      # Convert to lowercase
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove punctuation & numbers
    return text

df["clean_text"] = df["text"].apply(clean_text)

df.head()

,text,label,clean_text
0,Wall St. Bears Claw Back Into the Black (Reute...,2,wall st bears claw back into the black reuters...
1,Carlyle Looks Toward Commercial Aerospace (Reu...,2,carlyle looks toward commercial aerospace reut...
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,2,oil and economy cloud stocks outlook reuters r...
3,Iraq Halts Oil Exports from Main Southern Pipe...,2,iraq halts oil exports from main southern pipe...
4,"Oil prices soar to all-time record, posing new...",2,oil prices soar to alltime record posing new m...


Step 3: Tokenize Text

In [16]:
from tensorflow.keras.preprocessing.text import Tokenizer  #Imports the Tokenizer class(Tokenizer converts words into numbers.)

max_words = 10000

tokenizer = Tokenizer(num_words=max_words)  #Create a tokenizer object

tokenizer.fit_on_texts(df["clean_text"])   #Reads all the text and create a dictionary

sequences = tokenizer.texts_to_sequences(df["clean_text"])    #Converts every sentence into numbers

Step 4: Pad Sequences

In [18]:
from tensorflow.keras.preprocessing.sequence import pad_sequences    #import the padding function

max_length = 50    # every sentence will contain exactly 50 words

X = pad_sequences(
    sequences,
    maxlen=max_length,
    padding="post",         # Add zeros at the end of a sequence if it is shorter than maxlen
    truncating="post"       #Remove extra elements from the end of a sequence if it is longer than maxlen
)

print(X.shape)


(120000, 50)


Step 5: One-Hot Encode Labels

In [19]:
from tensorflow.keras.utils import to_categorical

y = to_categorical(df["label"])    #convert categorical data into a binary vector (0s and 1s)

print(y.shape)    #Displays label dimensions.

(120000, 4)


Step 6: Train-Test Split

In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

Step 7: Build the RNN Model

In [21]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense    #import 3 neural network

model = Sequential()    #Creates an empty neural network.

model.add(
    Embedding(
        input_dim=max_words,    #Vocabulary size
        output_dim=128,          #Each word becomes a 128-dimensional vector.
        input_length=max_length
    )
)

model.add(SimpleRNN(64))  #64 neurons(Learns sentence order and context.)

model.add(Dense(4, activation="softmax"))  #4 neurons, because AG News has four categories, Softmax converts outputs into probabilities.

model.summary()  #prints the neural network architecture

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Step 8: Compile the Model

In [10]:
model.compile(
    optimizer="adam",    #updates weights effiecently
    loss="categorical_crossentropy",   #Measures prediction error(used this becaus this is multiclass classification)
    metrics=["accuracy"]
)

Step 9: Train the Model

In [11]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,   #Entire dataset is processed 5 times.
    batch_size=64,  #64 samples are processed before updating weights.
    validation_split=0.2   #20% of training data is used for validation.
)

Epoch 1/5
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 48s 37ms/step - accuracy: 0.8082 - loss: 0.5808 - val_accuracy: 0.8661 - val_loss: 0.4584
Epoch 2/5
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 43s 36ms/step - accuracy: 0.8813 - loss: 0.3950 - val_accuracy: 0.8780 - val_loss: 0.4085
Epoch 3/5
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 45s 38ms/step - accuracy: 0.5877 - loss: 0.9137 - val_accuracy: 0.4303 - val_loss: 1.1785
Epoch 4/5
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 81s 37ms/step - accuracy: 0.4303 - loss: 1.1652 - val_accuracy: 0.4366 - val_loss: 1.1719
Epoch 5/5
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 83s 38ms/step - accuracy: 0.4340 - loss: 1.1609 - val_accuracy: 0.4097 - val_loss: 1.1841


Step 10: Evaluate the Model

In [12]:
loss, accuracy = model.evaluate(           #Tests the model using unseen data
    X_test,
    y_test
)

print("Test Accuracy:", accuracy)

750/750 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.4190 - loss: 1.1812
Test Accuracy: 0.4189999997615814


Step 11: Predict New News

In [13]:
sample = ["Apple launches a new AI-powered iPhone."]   #Creates a new news article.

sample = [clean_text(text) for text in sample]   #Cleans the sentence.

sample_seq = tokenizer.texts_to_sequences(sample)  #Converts words into numbers.

sample_pad = pad_sequences(   #Pads the sentence to 50 words.
    sample_seq,
    maxlen=max_length,
    padding="post"
)

prediction = model.predict(sample_pad)   #Predicts the class probabilities.

print(prediction)
print("Predicted Class:", prediction.argmax())   #Returns the class with the highest probability.

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 307ms/step
[[0.12269844 0.08108956 0.3549537  0.4412583 ]]
Predicted Class: 3
